In [7]:
import pandas as pd
import itertools
import os

def american_to_decimal(american_odds):
    """Convert American odds to decimal odds."""
    if american_odds > 0:
        return 1 + (american_odds / 100)
    else:
        return 1 + (100 / abs(american_odds))

def find_all_arbitrage(df):
    arbs = []

    # --- Normalize data ---
    df = df.copy()
    df["Outcome"] = df["Outcome"].astype(str).str.strip().str.lower()
    df["Bookmaker"] = df["Bookmaker"].astype(str).str.strip()
    df["Odds"] = pd.to_numeric(df["Odds"], errors="coerce")
    df = df.dropna(subset=["Odds"])

    grouped = df.groupby("Event")

    for event, group in grouped:
        outcomes = group["Outcome"].unique()
        num_outcomes = len(outcomes)

        if num_outcomes < 2:
            continue

        odds_by_outcome = {}

        for outcome in outcomes:
            odds_by_outcome[outcome] = group[group["Outcome"] == outcome][
                ["Bookmaker", "Odds"]
            ].to_records(index=False)

        outcome_combos = list(itertools.product(*odds_by_outcome.values()))

        for combo in outcome_combos:

            decimal_odds = []
            implied_prob = 0

            arb_row = {
                "Event": event,
                "Number of Outcomes": num_outcomes,
                "Outcomes": [],
                "Bookmakers": [],
                "Odds": [],
                "Recommended Stakes": [],
            }

            for (book, odds), outcome in zip(combo, odds_by_outcome.keys()):

                dec = american_to_decimal(odds)

                decimal_odds.append(dec)
                implied_prob += 1 / dec

                arb_row["Outcomes"].append(outcome)
                arb_row["Bookmakers"].append(book)
                arb_row["Odds"].append(odds)

            if implied_prob >= 1:
                continue

            stakes = [(1 / d) / implied_prob for d in decimal_odds]

            arb_row["Recommended Stakes"] = stakes
            arb_row["Total Probability"] = round(implied_prob, 6)
            arb_row["Profit Margin (%)"] = round((1 - implied_prob) * 100, 6)

            arb_row["Guaranteed Profit"] = round(
                min([s * d for s, d in zip(stakes, decimal_odds)]) - sum(stakes),
                6
            )

            arbs.append(arb_row)

    return arbs

def main():

    # Files stored directly in repo folder
    input_path = "Arbitrage_Input.xlsx"
    output_excel = "Arbitrage_Output.xlsx"
    output_csv = "Arbitrage_Output.csv"

    # Create template input file if it doesn't exist
    if not os.path.exists(input_path):

        sample_df = pd.DataFrame({
            "Event": ["Lakers vs Celtics", "Lakers vs Celtics"],
            "Outcome": ["Lakers", "Celtics"],
            "Bookmaker": ["DraftKings", "FanDuel"],
            "Odds": [120, -110]
        })

        sample_df.to_excel(input_path, index=False)

        print(f"Created template input file: {input_path}")
        print("Fill it with your sportsbook data and run again.")

        return

    # Read input
    df = pd.read_excel(input_path)

    required_cols = ["Event", "Outcome", "Bookmaker", "Odds"]

    if not all(c in df.columns for c in required_cols):
        raise ValueError(f"Missing required columns: {required_cols}")

    # Find arbitrage opportunities
    arbs = find_all_arbitrage(df)

    if not arbs:
        print("No arbitrage opportunities found.")
        return

    arb_df = pd.DataFrame(arbs)

    # Make lists readable
    arb_df["Outcomes"] = arb_df["Outcomes"].apply(
        lambda x: ", ".join(x)
    )

    arb_df["Bookmakers"] = arb_df["Bookmakers"].apply(
        lambda x: ", ".join(x)
    )

    arb_df["Odds"] = arb_df["Odds"].apply(
        lambda x: ", ".join(map(str, x))
    )

    arb_df["Recommended Stakes"] = arb_df["Recommended Stakes"].apply(
        lambda x: ", ".join([f"{round(v, 4)}" for v in x])
    )

    # Save outputs
    arb_df.to_excel(output_excel, index=False)
    arb_df.to_csv(output_csv, index=False)

    print(f"Excel output saved: {output_excel}")
    print(f"CSV output saved: {output_csv}")

if __name__ == "__main__":
    main()

Excel output saved: Arbitrage_Output.xlsx
CSV output saved: Arbitrage_Output.csv
